# ACTIVA M1 — real action predictions on 10 EPIC-100 train videos (M3 tuning data, #15 / #16)

**Settings:** GPU T4, Internet ON.

Runs `actions.detect_actions(info)` from `main` on the 10 train videos agreed on #15
(6 tune / 4 held-out, none in the Samsung manifest) and writes, per video, the scored predictions,
the ground truth mapped with the **same class-id -> key tables as the predictions**, source MD5 + SHA-256,
checkpoint SHA-256s, NMS/confidence settings and preprocessing/timebase metadata.

The official kit is private, so it is not uploaded here. Instead the engine's vocabulary check uses the EPIC
class keys directly; all 97 verbs / 300 nouns were verified to be in the kit vocabulary (#11).
Output: `/kaggle/working/m1_tuning/*.json` and `m1_tuning.zip`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!ffmpeg -version | head -1

In [ ]:
%cd /kaggle/working
!rm -rf ACTIVA dl ann
!git clone -q https://github.com/harshjha15335/ACTIVA.git && git -C ACTIVA log --oneline -1
!git clone -q https://github.com/epic-kitchens/epic-kitchens-download-scripts.git dl
!git clone -q https://github.com/epic-kitchens/epic-kitchens-100-annotations.git ann && git -C ann checkout -q ea8b40457a400c3fffa1c7f406ef3dc169cc2522 && git -C ann log --oneline -1
!pip install -q fvcore iopath simplejson pyyaml av gdown
!cd /kaggle/working/ACTIVA && python scripts/setup_m1_assets.py --dest /tmp/m1 > /kaggle/working/setup_m1.log 2>&1; tail -1 /kaggle/working/setup_m1.log

In [ ]:
TUNE = ["P01_06", "P02_04", "P08_06", "P21_03", "P24_04", "P26_21"]
HELD = ["P03_19", "P11_15", "P22_13", "P30_06"]
VIDEOS = TUNE + HELD
!cd /kaggle/working/dl && python epic_downloader.py --videos --specific-videos {",".join(VIDEOS)} --output-path /tmp/epic > /kaggle/working/dl.log 2>&1; tail -2 /kaggle/working/dl.log
import glob
MP4 = {v: (glob.glob(f"/tmp/epic/**/{v}.MP4", recursive=True) + glob.glob(f"/tmp/epic/**/{v}.mp4", recursive=True))[0] for v in VIDEOS}
print(MP4)

In [ ]:
import os, sys, csv, json, time, hashlib, dataclasses, pathlib
os.environ["ACTIVA_M1_ASSETS"] = "/tmp/m1"; os.environ["ACTIVA_DEVICE"] = "cuda"
sys.path[:0] = ["/kaggle/working/ACTIVA/code", "/kaggle/working/ACTIVA"]
import numpy as np, torch
import actions, video_io, action_config, slowfast_features as sf
from action_config import engine_assets, PAIRING, GRID
from slowfast_features import load_slowfast, Preprocess, FrameSource

A = engine_assets()
assert not A.missing(), A.missing()
VERB = actions.load_class_keys(A.classes_csv("verb")); NOUN = actions.load_class_keys(A.classes_csv("noun"))

class _Vocab:  # stand-in for vocab.load(): EPIC class keys (all present in the kit vocabulary, verified on #11)
    verbs, nouns, categories = tuple(VERB), tuple(NOUN), ()
    @staticmethod
    def is_valid_pair(v, n): return v in VERB and n in NOUN

actions._ENGINE.clear()
actions._ENGINE.update({
    "slowfast": load_slowfast(A.slowfast_code, A.slowfast_cfg, A.slowfast_ckpt, "cuda"),
    "verb": actions.ActionFormerRunner(A.af_code, A.af_cfg("verb"), actions._latest_ckpt(A.af_ckpt_dir("verb")), "cuda"),
    "noun": actions.ActionFormerRunner(A.af_code, A.af_cfg("noun"), actions._latest_ckpt(A.af_ckpt_dir("noun")), "cuda"),
    "verb_names": VERB, "noun_names": NOUN, "vocab": _Vocab,
})
ASSETS = json.load(open("/tmp/m1/ASSETS.json"))
from libs.core import load_config
AF_TEST = {k: load_config(str(A.af_cfg(k)))["test_cfg"] for k in ("verb", "noun")}
print("engine ready; NMS impl:", actions._ENGINE["verb"].nms_impl)

In [ ]:
def file_hashes(p):
    md5, sha = hashlib.md5(), hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 24), b""): md5.update(b); sha.update(b)
    return md5.hexdigest(), sha.hexdigest()

def ts(s):
    h, m, x = s.split(":"); return int(h) * 3600 + int(m) * 60 + float(x)

GT = {}
for r in csv.DictReader(open("/kaggle/working/ann/EPIC_100_train.csv", encoding="utf-8")):
    if r["video_id"] in VIDEOS:
        vi, ni = int(r["verb_class"]), int(r["noun_class"])
        GT.setdefault(r["video_id"], []).append({
            "segment": [round(ts(r["start_timestamp"]), 3), round(ts(r["stop_timestamp"]), 3)],
            "verb": VERB[vi], "noun": NOUN[ni], "verb_class": vi, "noun_class": ni,
            "narration_id": r["narration_id"], "narration": r["narration"]})
for v in VIDEOS: GT[v].sort(key=lambda g: g["segment"][0])
print({v: len(GT[v]) for v in VIDEOS})

In [ ]:
from eval.metrics import action_map  # M3's local scorer (proxy, not the official one)
OUT = pathlib.Path("/kaggle/working/m1_tuning"); OUT.mkdir(exist_ok=True)
SUMMARY = []
VI, NI = {k: i for i, k in enumerate(VERB)}, {k: i for i, k in enumerate(NOUN)}
for v in VIDEOS:
    md5, sha = file_hashes(MP4[v])
    info = video_io.inspect_video(MP4[v])
    torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    acts = actions.detect_actions(info)
    secs = time.time() - t0
    preds = [{"segment": [a.start, a.end], "verb": a.verb, "noun": a.noun, "score": round(float(a.score), 6),
              "verb_class": VI[a.verb], "noun_class": NI[a.noun]} for a in acts]
    m = action_map(GT[v], preds)
    rec = {
        "video_id": v, "split": "tune" if v in TUNE else "held-out",
        "source": {"file": os.path.basename(MP4[v]), "md5": md5, "sha256": sha, "duration_sec": info.duration,
                   "fps": info.fps, "width": info.width, "height": info.height, "rotation": info.rotation},
        "timebase": "D3: seconds = decoded PTS - stream start_time (video_io); times not trimmed or rebased",
        "engine": {"activa_commit": os.popen("git -C /kaggle/working/ACTIVA rev-parse HEAD").read().strip(),
                   "assets": ASSETS, "nms_impl": actions._ENGINE["verb"].nms_impl,
                   "frame_source": dataclasses.asdict(FrameSource()), "preprocess": dataclasses.asdict(Preprocess()),
                   "grid": dataclasses.asdict(GRID), "pairing": dataclasses.asdict(PAIRING),
                   "actionformer_test_cfg": AF_TEST, "device": torch.cuda.get_device_name(0)},
        "vocab_note": "class id -> key via EPIC_100_{verb,noun}_classes.csv for BOTH predictions and GT; all keys are in the kit vocabulary",
        "gt_source": "epic-kitchens-100-annotations@ea8b40457a400c3fffa1c7f406ef3dc169cc2522 EPIC_100_train.csv",
        "runtime": {"detect_actions_sec": round(secs, 1), "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2)},
        "local_proxy_mAP": {k: m[k]["mAP"] for k in ("pair", "verb", "noun")},
        "predictions": preds, "gt": GT[v],
    }
    json.dump(rec, open(OUT / f"{v}.json", "w"), indent=1)
    SUMMARY.append({k: rec[k] for k in ("video_id", "split")} | {"duration": round(info.duration, 1), "n_pred": len(preds),
                   "n_gt": len(GT[v]), "sec": round(secs, 1), **{f"mAP_{k}": round(x, 4) for k, x in rec["local_proxy_mAP"].items()},
                   "sha256": sha[:12]})
    print(SUMMARY[-1])

In [ ]:
for split, vids in (("tune", TUNE), ("held-out", HELD)):
    m = action_map({v: json.load(open(OUT / f"{v}.json"))["gt"] for v in vids},
                   {v: json.load(open(OUT / f"{v}.json"))["predictions"] for v in vids})
    print(split, {k: round(m[k]["mAP"], 4) for k in ("pair", "verb", "noun")})
json.dump(SUMMARY, open(OUT / "SUMMARY.json", "w"), indent=1)
!cd /kaggle/working && zip -qr m1_tuning.zip m1_tuning && ls -la m1_tuning.zip && sha256sum m1_tuning.zip